In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [24]:
def workflow_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Workflow improvement')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [25]:
workflow_deep_dive = workflow_research('2026-02-23','2026-02-24')

In [26]:
workflow_deep_dive

[{'event_id': '1350_2026-02-21',
  'output': "Microsoft added xAI's Grok 4.1 Fast to Microsoft Copilot Studio (preview). Practical impact: Copilot Studio now supports an additional fast-reasoning model option suitable for large-context, complex workflows and deeper tool use. Access is limited (preview; US-based makers initially), the feature is off by default and requires org admin enablement, and Grok is hosted outside Microsoft-managed infrastructure under xAI's terms (enterprise data-handling and contractual implications). For product, marketing, and design teams this means easier experimentation with a low-latency model inside Copilot Studio (faster prototyping, agent performance improvements) but also new admin/ procurement steps and data governance considerations.",
  'sources': [{'url': 'https://www.moneycontrol.com/technology/microsoft-adds-xai-s-grok-4-1-fast-to-copilot-studio-in-expanded-model-push-article-13838104.html',
    'name': 'Moneycontrol'}],
  'news_date': '2026-02-

In [27]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              {i['output']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [28]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()

In [29]:
openai_research_workflow(workflow_deep_dive[0:])

- REPORT SNAPSHOT - Publication: “More choice, more flexibility: xAI Grok 4.1 Fast now available in Microsoft Copilot Studio” by Ami Shastri, published Feb 19, 2026. [Source: Microsoft Copilot Blog, https://www.microsoft.com/en-us/microsoft-copilot/blog/copilot-studio/more-choice-more-flexibility-xai-grok-4-1-fast-now-available-in-microsoft-copilot-studio/]

- REPORT SNAPSHOT - What was published: Grok 4.1 Fast is a fast‑reasoning, text‑generation model (images not supported) designed for large context and deep tool use, to handle complex workflows in Copilot Studio. [Source: Microsoft Copilot Blog, https://www.microsoft.com/en-us/microsoft-copilot/blog/copilot-studio/more-choice-more-flexibility-xai-grok-4-1-fast-now-available-in-microsoft-copilot-studio/]

- REPORT SNAPSHOT - Rollout scope: Available in preview for United States–based makers; admin opt‑in required; readiness evaluations underway for other regions; customer data not retained or used to train xAI models; xAI-hosted out